# CardioSense — Recover the ECG models from the saved checkpoints

`04_ecg_feature_fusion.ipynb` saved **more than I first credited it with**. Its
`train_joint()` writes `joint_seed{seed}.pt` at the end of every epoch, so the
gated joint network survives a runtime disconnect — weights, optimiser state,
best-epoch weights, feature count and feature names.

What it does **not** write is the XGBoost booster, the fitted imputer and
scaler, or the per-class thresholds. So the position is:

| artifact | in the `.pt`? | recovery |
|---|---|---|
| `GatedJointNet` weights (best epoch) | **yes**, `state["best_state"]` | load |
| signal-only head (`signal_seresnet`) | **yes**, same weights | load |
| feature column order | **yes**, `state["feature_names"]` | load |
| best validation macro AUC, epoch count | **yes**, `state` | load |
| imputer / scaler | no | refit on train — deterministic, no training |
| per-class thresholds | no | re-tune on val — needs predictions |
| `features_xgb` booster (**the selected model**) | no | re-run one cell, 1–3 min |

### Two things worth knowing about the checkpoint format

**`ckpt["model"]` is not the best model.** The save happens every epoch, so that
key holds whatever the *last* epoch produced — including the epochs early
stopping had already rejected. The weights to use are
`ckpt["state"]["best_state"]`. Loading the wrong one costs accuracy and raises
no error anywhere, so `load_joint_checkpoint()` prefers `best_state` and records
which key it used.

**The signal branch has no preprocessing dependency.** `signal_logits()` runs
the encoder and its own auxiliary head and never enters the feature branch, so
waveform-only inference works from the `.pt` files and nothing else. That is the
one ECG path that works right now, from a cold runtime, with no further steps.

---

## How to run this

**Part A** (cells below, §1–§3) is a **cold start**: fresh runtime, no state from
notebook 04 needed. It loads the checkpoints and verifies them.

**Part B** (§4 onwards) needs notebook 04's feature pipeline in memory. That part
does **no training** — it is the download, join, split and preprocessing cells —
so the cheap route is:

1. Open `04_ecg_feature_fusion.ipynb`
2. Run its cells from the top **through §8** (the imputer/scaler cell)
3. Stop there. Do **not** run §9 onwards.
4. Run this notebook's Part B in that same kernel.

Part B then re-tunes the thresholds, saves the preprocessor, and optionally
re-runs the XGBoost cell to recover the selected model.

## 1. Setup — self-contained, nothing downloaded

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_DIR = Path("/content/CardioSense")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/artrai73/CardioSense.git", str(REPO_DIR)],
                       check=True)
    os.chdir(REPO_DIR)
    os.environ["CARDIOSENSE_DATA_ROOT"] = "/content/drive/MyDrive/CardioSense/data"
    DRIVE_ROOT = Path("/content/drive/MyDrive/CardioSense")
else:
    here = Path.cwd()
    while not (here / "pyproject.toml").exists() and here != here.parent:
        here = here.parent
    os.chdir(here)
    DRIVE_ROOT = Path(os.environ.get("CARDIOSENSE_DRIVE_ROOT", here / "models")).parent

SRC = Path.cwd() / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

MODELS_ECG = DRIVE_ROOT / "models" / "ecg"
MODELS_ECG.mkdir(parents=True, exist_ok=True)

print("repo       :", Path.cwd())
print("drive root :", DRIVE_ROOT)
print("ecg models :", MODELS_ECG)

## 2. Find and inspect the checkpoints

Before loading anything, read what each file claims about itself. `best_auc`
should match the `joint_gated` validation AUC printed in notebook 04's §13; if it
does not, these checkpoints came from a different run (a different
`USE_DEMOGRAPHICS` setting, or an older seed sweep) and the rest of this notebook
would be recovering the wrong model.

In [ ]:
import numpy as np
import pandas as pd

from cardiosense.fusion.ecg_joint import (
    CHECKPOINT_DIRS, ECG_CLASSES, find_joint_checkpoints, load_joint_ensemble,
)

paths = find_joint_checkpoints([DRIVE_ROOT, MODELS_ECG])

if not paths:
    raise FileNotFoundError(
        "No joint_seed*.pt found.\n"
        f"Searched {DRIVE_ROOT} and {MODELS_ECG}, each also with these "
        f"subdirectories appended: {list(CHECKPOINT_DIRS)}\n\n"
        "If your checkpoints are elsewhere, pass that directory directly:\n"
        "    paths = find_joint_checkpoints([Path('/content/drive/.../my_dir')])"
    )

print(f"found {len(paths)} checkpoint(s):")
for p in paths:
    print(f"  {p}  ({p.stat().st_size / 1024**2:.1f} MB)")

In [ ]:
checkpoints = load_joint_ensemble(paths)

table = pd.DataFrame([c.to_dict() for c in checkpoints])
display(table)

n_feat = checkpoints[0].n_feat
FEATURE_NAMES_FROM_CKPT = list(checkpoints[0].feature_names)

print(f"\nseeds loaded        : {len(checkpoints)}")
print(f"feature count       : {n_feat}")
print(f"feature names in .pt: {len(FEATURE_NAMES_FROM_CKPT)}")
print(f"weights key used    : {checkpoints[0].weights_key}")

if any(not c.finished for c in checkpoints):
    print("\n  NOTE: at least one seed did not reach `done` — it was interrupted "
          "mid-training. Its best_state is still the best epoch it reached, so it "
          "is usable, but it is not the model notebook 04 reported.")

if len(FEATURE_NAMES_FROM_CKPT) != n_feat:
    print(f"\n  WARNING: {len(FEATURE_NAMES_FROM_CKPT)} names for {n_feat} features.")

## 3. Verify the architecture actually matches

`load_state_dict` is strict here on purpose: it matches on parameter names and
shapes, so if the class in `cardiosense.fusion.ecg_joint` had drifted from the
one in the notebook — a renamed submodule, a reordered `Sequential`, a changed
default width — the load above would already have raised.

That it did not raise is the real verification. This cell confirms the loaded
weights also produce sane output, rather than a model that assembles and then
emits garbage.

In [ ]:
import torch

from cardiosense.fusion.ecg_joint import predict_signal_only, predict_joint

rng = np.random.default_rng(0)
fake_wave = rng.normal(0, 0.1, size=(12, 1000)).astype(np.float32)

p_signal = predict_signal_only(checkpoints, fake_wave)
print("signal-only probabilities on random input:")
for cls, p in zip(ECG_CLASSES, p_signal):
    print(f"  {cls:5s} {p:.4f}")

assert p_signal.shape == (len(ECG_CLASSES),), p_signal.shape
assert np.all((p_signal >= 0) & (p_signal <= 1)), "probabilities out of range"

# The joint path needs features too; random scaled features are enough to check
# the forward pass and the gate, not to interpret.
fake_feat = rng.normal(0, 1, size=n_feat).astype(np.float32)
p_joint, gate = predict_joint(checkpoints, fake_wave, fake_feat)
print(f"\njoint forward pass OK | mean gate toward signal = {gate:.3f}")
assert p_joint.shape == (len(ECG_CLASSES),)

# Transposed input must be accepted, not silently misread as 1000 channels.
p_t = predict_signal_only(checkpoints, fake_wave.T)
assert np.allclose(p_signal, p_t, atol=1e-5), "transpose handling is wrong"
print("(12, 1000) and (1000, 12) give identical output — orientation handled")

### The signal-only adapter works from here, with no further steps

At this point `ECGSignalAdapter` can run. It needs the `.pt` files and nothing
else. Its one gap is thresholds — see §5.

In [ ]:
from cardiosense.fusion import ECGSignalAdapter

signal_adapter = ECGSignalAdapter(MODELS_ECG, search_roots=[DRIVE_ROOT])
print("status:", signal_adapter.status())

prediction = signal_adapter.predict(fake_wave)
print(f"\nrisk (1 - P(NORM)) = {prediction.risk_score:.4f}")
print(f"calibrated         = {prediction.is_calibrated}")
print(f"reliability        = {prediction.reliability:.2f}  (capped: uncalibrated)")
print(f"threshold source   = {signal_adapter.metadata['threshold_source']}")

---

# Part B — needs notebook 04's feature pipeline in memory

Everything below requires the objects notebook 04 builds in §6–§8. **Run 04 from
the top through §8 and stop**, then continue here in the same kernel. Those cells
download two CSVs and fit an imputer; they train nothing.

## 4. Confirm the pipeline state is present

In [ ]:
required = {
    "X_train_raw": "raw train feature frame (§7)",
    "X_val_raw": "raw val feature frame (§7)",
    "X_test_raw": "raw test feature frame (§7)",
    "imputer": "SimpleImputer fitted on train (§8)",
    "scaler": "StandardScaler fitted on train (§8)",
    "FEATURE_NAMES": "training column order (§7b)",
    "y_val_feat": "val labels on the feature subset (§7)",
    "y_test_feat": "test labels on the feature subset (§7)",
    "CLASSES": "superclass names (§3)",
    "cfg": "the ecg config (§3)",
}
present = set(dir())
missing = [f"{k:14s} — {why}" for k, why in required.items() if k not in present]

if missing:
    raise RuntimeError(
        "Notebook 04's pipeline is not in this kernel:\n  "
        + "\n  ".join(missing)
        + "\n\nRun 04_ecg_feature_fusion.ipynb from the top through §8 "
          "(the imputer/scaler cell), stop there, then re-run this cell."
    )

print("pipeline state present")
print(f"  train {X_train_raw.shape}  val {X_val_raw.shape}  test {X_test_raw.shape}")
print(f"  features: {len(FEATURE_NAMES)}   classes: {list(CLASSES)}")

### Does the pipeline match the checkpoint?

This is the check that matters most in Part B. If `USE_DEMOGRAPHICS` differed
between the run that trained the checkpoint and this kernel, the feature count
differs, the first feature layer's shape differs, and every joint prediction
would be built from a feature vector the model never saw. The column *names* are
in the checkpoint, so this can be verified exactly rather than by count alone.

In [ ]:
mismatch = []
if len(FEATURE_NAMES) != n_feat:
    mismatch.append(f"count: pipeline {len(FEATURE_NAMES)} vs checkpoint {n_feat}")

if FEATURE_NAMES_FROM_CKPT:
    if list(FEATURE_NAMES) != FEATURE_NAMES_FROM_CKPT:
        only_pipeline = [c for c in FEATURE_NAMES if c not in set(FEATURE_NAMES_FROM_CKPT)]
        only_ckpt = [c for c in FEATURE_NAMES_FROM_CKPT if c not in set(FEATURE_NAMES)]
        if only_pipeline or only_ckpt:
            mismatch.append(f"names differ — only in pipeline: {only_pipeline[:6]}; "
                            f"only in checkpoint: {only_ckpt[:6]}")
        else:
            mismatch.append("same columns in a DIFFERENT ORDER — this alone would "
                            "scramble every feature the model reads")

if mismatch:
    raise RuntimeError(
        "The feature pipeline in this kernel does not match the checkpoint:\n  - "
        + "\n  - ".join(mismatch)
        + "\n\nMost often this is USE_DEMOGRAPHICS set differently than in the "
          "run that produced the .pt files. Set it to match and re-run 04 §6-§8."
    )

print("feature columns match the checkpoint exactly, in order")

## 5. Re-tune the per-class thresholds on validation

Not in the checkpoint, and 0.5 is wrong for every superclass — PTB-XL
prevalences run from roughly 12% (HYP) to 44% (NORM). They are re-derived with
`ev.tune_per_class_thresholds`, the same function `register()` used in notebook
04, on the same validation fold. **Val only.** Tuning on test would make every
test number afterwards meaningless.

In [ ]:
from cardiosense.ecg import evaluate as ev
from cardiosense.fusion.ecg_joint import predict_joint_batch, predict_signal_batch

# Waveforms for the feature subset, in the same record order as the features.
W_val = np.asarray(waveforms[splits["val"]["indices"][keep_val]], dtype=np.float32)
W_test = np.asarray(waveforms[splits["test"]["indices"][keep_test]], dtype=np.float32)
assert len(W_val) == len(X_val_raw), (len(W_val), len(X_val_raw))
print(f"waveforms aligned: val {W_val.shape}, test {W_test.shape}")

CLIP_SD = 5.0
X_val_s = np.clip(scaler.transform(imputer.transform(X_val_raw)), -CLIP_SD, CLIP_SD).astype(np.float32)
X_test_s = np.clip(scaler.transform(imputer.transform(X_test_raw)), -CLIP_SD, CLIP_SD).astype(np.float32)

p_val_signal = predict_signal_batch(checkpoints, W_val)
p_val_joint = predict_joint_batch(checkpoints, W_val, X_val_s)

thr_signal, _ = ev.tune_per_class_thresholds(y_val_feat, p_val_signal, cfg)
thr_joint, _ = ev.tune_per_class_thresholds(y_val_feat, p_val_joint, cfg)

print("\nthresholds re-tuned on val:")
print(pd.DataFrame({"class": list(CLASSES),
                    "signal_only": np.round(np.asarray(thr_signal, float), 4),
                    "joint_gated": np.round(np.asarray(thr_joint, float), 4)})
      .to_string(index=False))

In [ ]:
# Report test metrics so you can check these against notebook 04's §13 table.
p_test_signal = predict_signal_batch(checkpoints, W_test)
p_test_joint = predict_joint_batch(checkpoints, W_test, X_test_s)

rows = []
for name, p_test, thr in (("signal_seresnet", p_test_signal, thr_signal),
                          ("joint_gated", p_test_joint, thr_joint)):
    m = ev.evaluate_multilabel(y_test_feat, p_test, thr, cfg,
                               split_name=f"test/{name}", with_ci=False)
    rows.append({"model": name,
                 "test_macro_f1": round(float(m["macro_f1"]), 4),
                 "test_macro_auc": round(float(m["macro_roc_auc"]), 4)})

print(pd.DataFrame(rows).to_string(index=False))
print("\nCompare these with notebook 04 §13. A difference beyond ~0.002 means the")
print("checkpoints are from a different run than the numbers you are reporting.")

## 6. Save the preprocessor bundle

This is what the joint adapter is missing: the imputer's medians, the scaler's
means, the exact column order and the clip width. Without it the joint model
cannot be used at all, and with a *wrong* version of it the model runs and
returns confident nonsense.

In [ ]:
import json, pickle, time

preprocessor = {
    "imputer": imputer,
    "scaler": scaler,
    "feature_cols": list(FEATURE_NAMES),
    "clip_sd": CLIP_SD,
    "thresholds": np.asarray(thr_joint, dtype=float),
    "classes": list(CLASSES),
    "metadata": {
        "model_version": f"ecg-joint-{len(checkpoints)}seed-v1",
        "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "n_features": int(len(FEATURE_NAMES)),
        "checkpoints": [c.to_dict() for c in checkpoints],
        "thresholds_tuned_on": "validation fold (fold 9)",
        "is_calibrated": False,
    },
}

preprocessor_path = MODELS_ECG / "ecg_feature_preprocessor.pkl"
with open(preprocessor_path, "wb") as handle:
    pickle.dump(preprocessor, handle)

# Thresholds for the waveform-only adapter, which needs no preprocessor at all.
(MODELS_ECG / "ecg_signal_thresholds.json").write_text(json.dumps({
    "classes": list(CLASSES),
    "thresholds": [round(float(t), 6) for t in np.asarray(thr_signal, float)],
    "source": "re-tuned on validation fold by 04c",
}, indent=2), encoding="utf-8")

print("saved:")
print(" ", preprocessor_path)
print(" ", MODELS_ECG / "ecg_signal_thresholds.json")

## 7. Recover `features_xgb` — the model notebook 04 actually selected

Notebook 04 chose its final model by validation macro AUC, and that was
`features_xgb` at macro F1 **0.771** / macro AUC **0.9413**. The boosters were
never serialised, so this re-runs that one cell. On a T4 it is 1–3 minutes; the
hyperparameters, seed and early-stopping rule are copied from §10 unchanged, so
the result should reproduce.

**It trains on the raw frame.** `X_train_raw`, not `X_train` — XGBoost learns a
default direction for each missing value, and imputing first throws that away.
The saved bundle records `preprocessing="raw"` so the adapter cannot get this
wrong later: feeding scaled, imputed values to this booster raises no error and
silently degrades every prediction.

Set `RECOVER_XGB = False` to skip and rely on the joint model instead.

In [ ]:
RECOVER_XGB = True

if RECOVER_XGB:
    import time as _time
    from sklearn.metrics import roc_auc_score
    try:
        import xgboost as xgb
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0"],
                       check=True)
        import xgboost as xgb

    device_type = "cuda" if torch.cuda.is_available() else "cpu"
    XGB_PARAMS = dict(
        n_estimators=4000, learning_rate=0.02, max_depth=6, min_child_weight=2,
        subsample=0.8, colsample_bytree=0.4, reg_lambda=2.0, tree_method="hist",
        eval_metric="auc", early_stopping_rounds=200, random_state=int(cfg.seed),
    )
    if int(xgb.__version__.split(".")[0]) >= 2:
        XGB_PARAMS["device"] = device_type

    Xtr_raw = X_train_raw.to_numpy(np.float32)
    Xva_raw = X_val_raw.to_numpy(np.float32)
    Xte_raw = X_test_raw.to_numpy(np.float32)

    start = _time.time()
    xgb_models = []
    val_prob_xgb = np.zeros_like(y_val_feat)
    test_prob_xgb = np.zeros_like(y_test_feat)
    for c, name in enumerate(CLASSES):
        clf = xgb.XGBClassifier(**XGB_PARAMS)
        clf.fit(Xtr_raw, y_train_feat[:, c], eval_set=[(Xva_raw, y_val_feat[:, c])],
                verbose=False)
        val_prob_xgb[:, c] = clf.predict_proba(Xva_raw)[:, 1]
        test_prob_xgb[:, c] = clf.predict_proba(Xte_raw)[:, 1]
        xgb_models.append(clf)
        print(f"  {name:5s}: best iteration {clf.best_iteration:4d} | val AUC "
              f"{roc_auc_score(y_val_feat[:, c], val_prob_xgb[:, c]):.4f}")
    print(f"\n{_time.time() - start:.0f}s")

    thr_xgb, _ = ev.tune_per_class_thresholds(y_val_feat, val_prob_xgb, cfg)
    m_xgb = ev.evaluate_multilabel(y_test_feat, test_prob_xgb, thr_xgb, cfg,
                                   split_name="test/features_xgb", with_ci=False)
    print(f"\nfeatures_xgb  test macro F1 {m_xgb['macro_f1']:.4f} | "
          f"test macro AUC {m_xgb['macro_roc_auc']:.4f}")
    print("notebook 04 reported  macro F1 0.7710 | macro AUC 0.9413")
else:
    xgb_models = None
    print("skipped")

In [ ]:
if RECOVER_XGB:
    bundle = {
        "model": xgb_models,                 # five one-vs-rest binary boosters
        "preprocessing": "raw",              # honoured by ECGAdapter -- see docstring
        "imputer": None,
        "scaler": None,
        "feature_cols": list(FEATURE_NAMES),
        "classes": list(CLASSES),
        "thresholds": np.asarray(thr_xgb, dtype=float),
        "clip_sd": CLIP_SD,
        "metadata": {
            "model_version": "ecg-features_xgb-v1",
            "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "selected_model": "features_xgb",
            "selected_by": "validation macro AUC (notebook 04 §13)",
            "n_features": int(len(FEATURE_NAMES)),
            "test_macro_f1": round(float(m_xgb["macro_f1"]), 4),
            "test_macro_auc": round(float(m_xgb["macro_roc_auc"]), 4),
            "thresholds_tuned_on": "validation fold (fold 9)",
            "is_calibrated": False,
            "note": ("Trained on the raw feature frame with NaNs intact. Do not "
                     "impute or scale before predict_proba."),
        },
    }
    bundle_path = MODELS_ECG / "ecg_feature_model.pkl"
    with open(bundle_path, "wb") as handle:
        pickle.dump(bundle, handle)
    print("saved:", bundle_path)

## 8. Verify every saved artifact reloads and reproduces its predictions

Saving a silently wrong artifact is worse than saving nothing. Each bundle is
reloaded from disk through the adapters — not through the objects still in
memory — and checked against the in-memory predictions on real records.

In [ ]:
from cardiosense.fusion import ECGAdapter, ECGJointAdapter, resolve_ecg_adapter

results = []

# -- joint adapter: .pt + preprocessor
joint_adapter = ECGJointAdapter(MODELS_ECG, search_roots=[DRIVE_ROOT])
assert joint_adapter.available, joint_adapter.load_error

row = 0
raw_row = X_val_raw.iloc[row].to_dict()
pred = joint_adapter.predict({"waveform": W_val[row], "features": raw_row})
expected_risk = float(1.0 - p_val_joint[row, list(CLASSES).index("NORM")])
results.append(("joint_gated", pred.risk_score, expected_risk))

# -- signal adapter: .pt only, now with tuned thresholds
signal_adapter = ECGSignalAdapter(MODELS_ECG, search_roots=[DRIVE_ROOT])
pred_s = signal_adapter.predict(W_val[row])
expected_s = float(1.0 - p_val_signal[row, list(CLASSES).index("NORM")])
results.append(("signal_seresnet", pred_s.risk_score, expected_s))
print("signal threshold source:", signal_adapter.metadata["threshold_source"])

# -- feature adapter: the selected XGBoost model
if RECOVER_XGB:
    feat_adapter = ECGAdapter(MODELS_ECG)
    assert feat_adapter.available, feat_adapter.load_error
    assert feat_adapter.preprocessing == "raw", feat_adapter.preprocessing
    pred_f = feat_adapter.predict(raw_row)
    expected_f = float(1.0 - val_prob_xgb[row, list(CLASSES).index("NORM")])
    results.append(("features_xgb", pred_f.risk_score, expected_f))

print()
for name, got, want in results:
    ok = abs(got - want) < 1e-4
    print(f"  {name:18s} adapter {got:.6f}  in-memory {want:.6f}  "
          f"{'match' if ok else 'MISMATCH'}")
    assert ok, f"{name}: adapter and in-memory predictions disagree"

print("\nall reloaded adapters reproduce their in-memory predictions")

In [ ]:
# What the engine will actually pick up, and under which name.
chosen = resolve_ecg_adapter(MODELS_ECG, search_roots=[DRIVE_ROOT])
print("resolved ECG adapter:", type(chosen).__name__)
print("variant             :", chosen.variant)
print("model_version       :", chosen.status()["model_version"])
print("calibrated          :", chosen.is_calibrated,
      f"-> reliability capped at {chosen.reliability:.2f}")

---

## Done

| file | what it unlocks |
|---|---|
| `ecg_signal_thresholds.json` | tuned labels for the waveform-only adapter |
| `ecg_feature_preprocessor.pkl` | the joint (waveform + features) adapter |
| `ecg_feature_model.pkl` | `features_xgb`, the model notebook 04 selected |

`resolve_ecg_adapter()` prefers them in that reverse order — the selected
feature model first, the joint model next, the waveform-only head as the
fallback that always works — and `status()["model_version"]` names whichever ran,
so a report can never imply the selected model produced a number the fallback
produced.

**Download copies to your own machine.** Drive is one accidental deletion from
costing a retrain, and the boosters above took 1–3 minutes only because the
feature CSVs were already there.

Still outstanding for ECG: **nothing here is calibrated.** Every adapter reports
`is_calibrated=False` and reliability capped at 0.50, which is correct and is
what the fusion layer assumes. Running `cardiosense.ecg.calibrate` would lift
that cap honestly; until then the ECG contribution is a ranking score, and the
fusion report says so.

Next: `05_multimodal_fusion.ipynb`.